# Grad-CAM — HEALTH '다이어트' (34- vs 65+)

`URP_논문/gradcam/gradcam_v4.ipynb`(EDU '심리와 학습')를 다이어트 표본에 맞게 튜닝한 버전.

- 입력: `../00_sample/sampling.py`의 `load_diet_sample()` — 원본 v3 CSV에서 시드 42 고정 샘플링 (34-/65+ 각 1,250개, 매번 동일한 2,500행 재현)
- 표본이 이미 균형이므로 원본의 undersampling 로직 제거, train/test 8:2 분할만 유지
- 모델: DINOv2-base 백본 고정 + 분류 헤드 학습 → Grad-CAM → ROI(텍스트/인물/배경) 정량 검증
- 모든 산출물은 `outputs/`에 저장됨
- 환경: conda `urp_yena` (torch 2.11+cu128, grad-cam, transformers, easyocr, ultralytics 설치 확인됨)


In [1]:
# ── 경로 설정 ─────────────────────────────────────────────────────────────────
import os
from pathlib import Path

NB_DIR   = Path.cwd() if "__file__" not in dir() else Path(__file__).parent
# 노트북 실행 시 cwd가 01_gradcam이라고 가정. 아니면 아래를 절대경로로 수정.
STAGE_DIR = Path("/home/urp_jwl/URP_backup/26-1_URP/7_HEALTH_다이어트_분석/01_gradcam")
BASE_DIR  = STAGE_DIR.parent.parent          # 26-1_URP
SAMPLE_DIR = STAGE_DIR.parent / "00_sample"  # 공용 샘플링 모듈 위치
OUT_DIR    = STAGE_DIR / "outputs"
YOLO_WEIGHTS = BASE_DIR / "6_썸네일_attention_gradcam/yolov8n.pt"

OUT_DIR.mkdir(exist_ok=True)
os.chdir(OUT_DIR)   # 이후 모든 상대경로 저장물(png/csv/pkl)이 outputs/에 쌓임

import sys
sys.path.insert(0, str(SAMPLE_DIR))

print("sample :", SAMPLE_DIR / "sampling.py")
print("outputs:", OUT_DIR)
print("yolo   :", YOLO_WEIGHTS, YOLO_WEIGHTS.exists())


sample : /home/urp_jwl/URP_backup/26-1_URP/7_HEALTH_다이어트_분석/00_sample/sampling.py
outputs: /home/urp_jwl/URP_backup/26-1_URP/7_HEALTH_다이어트_분석/01_gradcam/outputs
yolo   : /home/urp_jwl/URP_backup/26-1_URP/6_썸네일_attention_gradcam/yolov8n.pt True


In [2]:
# ── 데이터 로드 (공용 샘플링 모듈, 시드 42 고정 → 항상 동일한 2,500행) ─────────
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sampling import load_diet_sample

RANDOM_STATE = 42

df = load_diet_sample()   # 썸네일 절대경로(resolved_path) 포함, 존재 확인 완료
print(f"표본: {len(df)}행 / age_group: {df['age_group'].value_counts().to_dict()}")

# ── 라벨: age_group -> target (34-=0, 65+=1) ──────────────────────────────────
# 원본 노트북의 y_grouped 표기('~34'/'65~')를 그대로 사용해 이후 셀과 호환
df["y_grouped"] = df["age_group"].map({"34-": "~34", "65+": "65~"})
df["target"]    = df["y_grouped"].map({"~34": 0, "65~": 1})

# 표본이 이미 1,250/1,250 균형이므로 별도 undersampling 없이 바로 분할
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df["target"]
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {train_df['y_grouped'].value_counts().to_dict()}")
print(f"Test : {test_df['y_grouped'].value_counts().to_dict()}")


표본: 2500행 / age_group: {'34-': 1250, '65+': 1250}
Train: {'65~': 1000, '~34': 1000}
Test : {'~34': 250, '65~': 250}


/home/urp_jwl/URP_backup/26-1_URP/7_HEALTH_다이어트_분석/00_sample/sampling.py:54: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=N_PER_GROUP, random_state=SEED))


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, AutoModel

MODEL_NAME = "facebook/dinov2-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)


class ThumbDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["resolved_path"]).convert("RGB")
        x = self.processor(images=img, return_tensors="pt")
        pixel_values = x["pixel_values"].squeeze(0)
        y = int(row["target"])
        return pixel_values, y, row["resolved_path"], row["y_grouped"]


def collate_fn(batch):
    pixel_values = torch.stack([b[0] for b in batch])
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    paths = [b[2] for b in batch]
    groups = [b[3] for b in batch]
    return pixel_values, labels, paths, groups


class DinoBinaryClassifier(nn.Module):
    def __init__(self, model_name="facebook/dinov2-base", num_classes=2):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(
            model_name,
            attn_implementation="eager"  # required for GradCAM gradient flow
        )
        hidden = self.backbone.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values, return_dict=True)
        cls = outputs.last_hidden_state[:, 0, :]   # [CLS] token
        logits = self.classifier(cls)
        return logits


model = DinoBinaryClassifier(MODEL_NAME).to(DEVICE)

train_ds = ThumbDataset(train_df, processor)
test_ds  = ThumbDataset(test_df,  processor)

USE_CUDA = torch.cuda.is_available()

train_loader = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=USE_CUDA
)

test_loader = DataLoader(
    test_ds,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=USE_CUDA
)

print("CUDA available:", USE_CUDA)
print("train_loader num_workers:", train_loader.num_workers)
print("test_loader num_workers:", test_loader.num_workers)


Using device: cuda


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

CUDA available: True
train_loader num_workers: 0
test_loader num_workers: 0


In [4]:
# Freeze backbone — train classifier head only
for p in model.backbone.parameters():
    p.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for pixel_values, labels, _, _ in train_loader:
        pixel_values = pixel_values.to(DEVICE)
        labels       = labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(pixel_values)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"[epoch {epoch+1}/{EPOCHS}] loss={total_loss/len(train_loader):.4f}")

# Re-enable gradients for the last encoder block (needed for GradCAM)
for p in model.backbone.encoder.layer[-1].parameters():
    p.requires_grad = True

print("Training complete. Last encoder block gradients enabled for GradCAM.")


[epoch 1/5] loss=0.5985
[epoch 2/5] loss=0.3823
[epoch 3/5] loss=0.2667
[epoch 4/5] loss=0.1697
[epoch 5/5] loss=0.1224
Training complete. Last encoder block gradients enabled for GradCAM.


In [5]:
import numpy as np
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for pixel_values, labels, _, _ in test_loader:
        pixel_values = pixel_values.to(DEVICE)
        logits = model(pixel_values)
        pred   = torch.argmax(logits, dim=1).cpu().numpy()
        y_true.extend(labels.numpy())
        y_pred.extend(pred)

print("accuracy        :", accuracy_score(y_true, y_pred))
print("balanced_accuracy:", balanced_accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=["~34", "65~"]))


accuracy        : 0.718
balanced_accuracy: 0.718
              precision    recall  f1-score   support

         ~34       0.68      0.81      0.74       250
         65~       0.77      0.62      0.69       250

    accuracy                           0.72       500
   macro avg       0.73      0.72      0.72       500
weighted avg       0.73      0.72      0.72       500



In [6]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

FONT_PATH = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

# Matplotlib에 폰트 등록
fm.fontManager.addfont(FONT_PATH)

korean_font = fm.FontProperties(fname=FONT_PATH)
font_name = korean_font.get_name()

# 전체 그래프 기본 폰트 설정
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("한글 폰트 설정 완료:", font_name)

한글 폰트 설정 완료: NanumGothic


In [7]:
import numpy as np
import matplotlib
matplotlib.use("Agg")   # headless GPU server
import matplotlib.pyplot as plt
from PIL import Image

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image


def reshape_transform(tensor):
    """
    ViT hidden state [B, 1+N_patches, C] -> [B, C, H, W]
    DINOv2-base: 224px input -> 14x14 = 196 patches
    """
    tensor = tensor[:, 1:, :]           # remove CLS token
    n_patches = tensor.size(1)
    h = w = int(n_patches ** 0.5)
    assert h * w == n_patches, f"Patch count ({n_patches}) is not square."
    tensor = tensor.reshape(tensor.size(0), h, w, tensor.size(2))
    tensor = tensor.permute(0, 3, 1, 2).contiguous()   # [B, C, H, W]
    return tensor


# Use norm1 of the last Transformer block as the target layer
target_layers = [model.backbone.encoder.layer[-1].norm1]

cam = GradCAM(
    model=model,
    target_layers=target_layers,
    reshape_transform=reshape_transform,
)

CLASS_NAMES = {0: "~34", 1: "65~"}
print("GradCAM initialized successfully")


GradCAM initialized successfully


In [8]:
def run_gradcam_on_image(image_path, save_path=None):
    """
    Visualize Grad-CAM for a single thumbnail.
    Layout: [Original] [GradCAM->~34] [Heatmap ~34] [GradCAM->65~] [Heatmap 65~]
    """
    model.eval()

    img_orig = Image.open(image_path).convert("RGB")
    inputs = processor(images=img_orig, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(DEVICE)

    # Prediction
    with torch.no_grad():
        logits = model(pixel_values)
        probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()
    pred_class = int(np.argmax(probs))

    # Prepare input image array
    _, _, H, W = pixel_values.shape
    img_resized = img_orig.resize((W, H))
    rgb_img = np.array(img_resized).astype(np.float32) / 255.0

    # Compute GradCAM for both classes
    cam_young = cam(input_tensor=pixel_values,
                    targets=[ClassifierOutputTarget(0)])[0]   # ~34 focus
    cam_old   = cam(input_tensor=pixel_values,
                    targets=[ClassifierOutputTarget(1)])[0]   # 65~ focus

    vis_young = show_cam_on_image(rgb_img, cam_young, use_rgb=True)
    vis_old   = show_cam_on_image(rgb_img, cam_old,   use_rgb=True)

    fig, axes = plt.subplots(1, 5, figsize=(25, 5))

    axes[0].imshow(img_resized)
    axes[0].set_title(
        f"Original\npred={CLASS_NAMES[pred_class]}  "
        f"({probs[0]*100:.1f}% ~34 / {probs[1]*100:.1f}% 65~)"
    )
    axes[0].axis("off")

    axes[1].imshow(vis_young)
    axes[1].set_title("GradCAM -> ~34 (young)\nRegion attended by ~34")
    axes[1].axis("off")

    im0 = axes[2].imshow(cam_young, cmap="jet", vmin=0, vmax=1)
    axes[2].set_title("Heatmap (~34)")
    axes[2].axis("off")
    plt.colorbar(im0, ax=axes[2], fraction=0.046)

    axes[3].imshow(vis_old)
    axes[3].set_title("GradCAM -> 65~ (old)\nRegion attended by 65~")
    axes[3].axis("off")

    im1 = axes[4].imshow(cam_old, cmap="jet", vmin=0, vmax=1)
    axes[4].set_title("Heatmap (65~)")
    axes[4].axis("off")
    plt.colorbar(im1, ax=axes[4], fraction=0.046)

    plt.suptitle(os.path.basename(image_path), fontsize=10)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved: {save_path}")
    else:
        plt.show()
    plt.close(fig)

# Test on one image
image_path = test_df.iloc[0]["resolved_path"]
run_gradcam_on_image(image_path, save_path="gradcam_single.png")
print("Done")


Saved: gradcam_single.png
Done


In [9]:
def show_gradcam_grid(
    df_sub,
    max_images=10,
    save_path="gradcam_grid.png"
):
    """
    Grid visualization of Grad-CAM for multiple thumbnails.
    Each row: [Original] [GradCAM->~34] [Heatmap ~34] [GradCAM->65~] [Heatmap 65~]
    """
    df_sub = df_sub.head(max_images).reset_index(drop=True)
    n = len(df_sub)

    fig, axes = plt.subplots(n, 5, figsize=(25, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    model.eval()
    for i, (_, row) in enumerate(df_sub.iterrows()):
        img_path = row["resolved_path"]
        img_orig = Image.open(img_path).convert("RGB")

        inputs = processor(images=img_orig, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(DEVICE)

        _, _, H, W = pixel_values.shape
        img_resized = img_orig.resize((W, H))
        rgb_img = np.array(img_resized).astype(np.float32) / 255.0

        with torch.no_grad():
            logits = model(pixel_values)
            probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()
        pred_class = int(np.argmax(probs))

        cam_young = cam(input_tensor=pixel_values,
                        targets=[ClassifierOutputTarget(0)])[0]
        cam_old   = cam(input_tensor=pixel_values,
                        targets=[ClassifierOutputTarget(1)])[0]

        vis_young = show_cam_on_image(rgb_img, cam_young, use_rgb=True)
        vis_old   = show_cam_on_image(rgb_img, cam_old,   use_rgb=True)

        correct = (pred_class == int(row["target"]))
        flag = "O" if correct else "X"

        axes[i, 0].imshow(img_resized)
        axes[i, 0].set_title(
            f"{flag} true={row['y_grouped']} | pred={CLASS_NAMES[pred_class]}\n"
            f"{probs[0]*100:.1f}% ~34 / {probs[1]*100:.1f}% 65~"
        )
        axes[i, 0].axis("off")

        axes[i, 1].imshow(vis_young)
        axes[i, 1].set_title("GradCAM -> ~34\n(young attention)")
        axes[i, 1].axis("off")

        im0 = axes[i, 2].imshow(cam_young, cmap="jet", vmin=0, vmax=1)
        axes[i, 2].set_title("Heatmap (~34)")
        axes[i, 2].axis("off")
        plt.colorbar(im0, ax=axes[i, 2], fraction=0.046)

        axes[i, 3].imshow(vis_old)
        axes[i, 3].set_title("GradCAM -> 65~\n(old attention)")
        axes[i, 3].axis("off")

        im1 = axes[i, 4].imshow(cam_old, cmap="jet", vmin=0, vmax=1)
        axes[i, 4].set_title("Heatmap (65~)")
        axes[i, 4].axis("off")
        plt.colorbar(im1, ax=axes[i, 4], fraction=0.046)

    plt.suptitle("Grad-CAM -- ~34 vs 65~ Attended Region Comparison", fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")
    plt.close(fig)

show_gradcam_grid(test_df.sample(10, random_state=42), max_images=10)


Saved: gradcam_grid.png


In [10]:
# ── Collect predictions ───────────────────────────────────────────────────────
model.eval()
all_preds, all_probs, all_paths, all_true = [], [], [], []

with torch.no_grad():
    for pixel_values, labels, paths, _ in test_loader:
        pixel_values = pixel_values.to(DEVICE)
        logits = model(pixel_values)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = np.argmax(probs, axis=1)
        all_preds.extend(preds)
        all_probs.extend(probs)
        all_paths.extend(paths)
        all_true.extend(labels.numpy())

result_df = test_df.copy().reset_index(drop=True)
result_df["pred"]    = all_preds
result_df["prob_65"] = [p[1] for p in all_probs]
result_df["correct"] = (result_df["target"] == result_df["pred"])

# ── Auto-detect channel column ────────────────────────────────────────────────
CHANNEL_COL_CANDIDATES = ["channel_id", "channelId", "channel", "channel_name"]
channel_col = None
for c in CHANNEL_COL_CANDIDATES:
    if c in result_df.columns:
        channel_col = c
        break

def dedup_by_channel(df_sorted, top_n=50):
    if channel_col is None:
        print("  WARNING: No channel column found -> returning top rows without dedup")
        return df_sorted.head(top_n)
    seen_channels = set()
    selected = []
    for _, row in df_sorted.iterrows():
        ch = row[channel_col]
        if ch not in seen_channels:
            seen_channels.add(ch)
            selected.append(row)
        if len(selected) == top_n:
            break
    result = pd.DataFrame(selected).reset_index(drop=True)
    print(f"  After channel dedup: {len(result)} selected ({result[channel_col].nunique()} channels)")
    return result

TOP_N = 50

# ── Correct top50 — 65~ high confidence ───────────────────────────────────────
print("=== Correct samples (65~) ===")
correct_65_sorted  = result_df[(result_df["correct"]) & (result_df["target"] == 1)] \
                        .sort_values("prob_65", ascending=False)
correct_samples    = dedup_by_channel(correct_65_sorted, top_n=TOP_N)

# ── Correct top50 — ~34 high confidence ───────────────────────────────────────
print("=== Correct samples (~34) ===")
correct_34_sorted  = result_df[(result_df["correct"]) & (result_df["target"] == 0)] \
                        .sort_values("prob_65", ascending=True)
correct_34_samples = dedup_by_channel(correct_34_sorted, top_n=TOP_N)

# ── Incorrect top50 ────────────────────────────────────────────────────────────
print("=== Incorrect samples ===")
incorrect_all    = result_df[~result_df["correct"]].copy()
incorrect_all["conf"] = incorrect_all.apply(
    lambda r: r["prob_65"] if r["pred"] == 1 else (1 - r["prob_65"]), axis=1
)
incorrect_sorted  = incorrect_all.sort_values("conf", ascending=False)
incorrect_samples = dedup_by_channel(incorrect_sorted, top_n=TOP_N)

# ── GradCAM visualization ──────────────────────────────────────────────────────
show_gradcam_grid(correct_samples,    max_images=TOP_N, save_path="gradcam_correct_65_top50.png")
show_gradcam_grid(correct_34_samples, max_images=TOP_N, save_path="gradcam_correct_34_top50.png")
if len(incorrect_samples) > 0:
    show_gradcam_grid(incorrect_samples, max_images=TOP_N, save_path="gradcam_incorrect_top50.png")

# ── Save CSV ───────────────────────────────────────────────────────────────────
SAVE_COLS = ["resolved_path", "channel_name", "channel_id", "title", "video_id",
             "target", "pred", "prob_65", "correct"]
save_cols = [c for c in SAVE_COLS if c in correct_samples.columns]

correct_samples[save_cols].to_csv("samples_correct_65_top50.csv", index=False, encoding="utf-8-sig")
correct_34_samples[save_cols].to_csv("samples_correct_34_top50.csv", index=False, encoding="utf-8-sig")
if len(incorrect_samples) > 0:
    incorrect_samples[save_cols].to_csv("samples_incorrect_top50.csv", index=False, encoding="utf-8-sig")

print("\nDone.")
print(f"  gradcam_correct_65_top50.png  + samples_correct_65_top50.csv  ({len(correct_samples)} images)")
print(f"  gradcam_correct_34_top50.png  + samples_correct_34_top50.csv  ({len(correct_34_samples)} images)")
print(f"  gradcam_incorrect_top50.png   + samples_incorrect_top50.csv   ({len(incorrect_samples)} images)")


=== Correct samples (65~) ===
  After channel dedup: 50 selected (50 channels)
=== Correct samples (~34) ===
  After channel dedup: 50 selected (50 channels)
=== Incorrect samples ===
  After channel dedup: 50 selected (50 channels)
Saved: gradcam_correct_65_top50.png
Saved: gradcam_correct_34_top50.png
Saved: gradcam_incorrect_top50.png

Done.
  gradcam_correct_65_top50.png  + samples_correct_65_top50.csv  (50 images)
  gradcam_correct_34_top50.png  + samples_correct_34_top50.csv  (50 images)
  gradcam_incorrect_top50.png   + samples_incorrect_top50.csv   (50 images)


In [11]:
# =============================================================================
# Quantitative Analysis -- Grad-CAM Reliability Validation
#   ROI auto-detection: EasyOCR (text) + YOLOv8 (person) -> background (rest)
#   1. ROI mask generation & caching
#   2. ROI-wise Deletion / Insertion bar charts
#   3. Masking visualization grid (50 images -> 2 files)
#   4. Full Deletion / Insertion curves
# =============================================================================
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
import torch, os, pickle
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

import easyocr
from ultralytics import YOLO

print("Initializing EasyOCR (Korean + English)...")
ocr_reader = easyocr.Reader(["ko", "en"], gpu=torch.cuda.is_available(), verbose=False)
print("Initializing YOLOv8...")
yolo_model = YOLO(str(YOLO_WEIGHTS))
print("Models loaded")

# ── Helpers ───────────────────────────────────────────────────────────────────
def get_prob(pixel_values):
    model.eval()
    with torch.no_grad():
        logits = model(pixel_values)
        return torch.softmax(logits, dim=1)[0].cpu().numpy()

def get_pixel_values(img_pil):
    inputs = processor(images=img_pil, return_tensors="pt")
    pv     = inputs["pixel_values"].to(DEVICE)
    _, _, H, W = pv.shape
    img_resized = img_pil.resize((W, H))
    rgb_np = np.array(img_resized).astype(np.float32) / 255.0
    return pv, rgb_np, H, W

def get_cam(pixel_values, target_class):
    return cam(
        input_tensor=pixel_values,
        targets=[ClassifierOutputTarget(target_class)]
    )[0]

# ── ROI mask generation ───────────────────────────────────────────────────────
ROI_CACHE_FILE = "roi_masks_cache.pkl"

def build_roi_masks(image_path, H, W):
    """
    Build ROI masks at (H, W) resolution.
    Returns: {"text": bool[H,W], "person": bool[H,W], "background": bool[H,W]}
    """
    img_orig = Image.open(image_path).convert("RGB")
    orig_W, orig_H = img_orig.size
    img_np = np.array(img_orig)

    mask_text   = np.zeros((orig_H, orig_W), dtype=bool)
    mask_person = np.zeros((orig_H, orig_W), dtype=bool)

    # Text region (EasyOCR)
    ocr_results = ocr_reader.readtext(
        img_np, detail=1, paragraph=False,
        min_size=10, text_threshold=0.4, low_text=0.3,
        link_threshold=0.3, width_ths=0.8,
        contrast_ths=0.05, adjust_contrast=0.7,
    )
    for bbox, text, conf in ocr_results:
        if conf < 0.3:
            continue
        pts = np.array(bbox, dtype=np.int32)
        x0, y0 = pts[:, 0].min(), pts[:, 1].min()
        x1, y1 = pts[:, 0].max(), pts[:, 1].max()
        x0, y0 = max(0, x0), max(0, y0)
        x1, y1 = min(orig_W, x1), min(orig_H, y1)
        mask_text[y0:y1, x0:x1] = True

    # Person region (YOLOv8 class 0)
    yolo_results = yolo_model(img_np, verbose=False)[0]
    for box in yolo_results.boxes:
        cls_id = int(box.cls[0])
        conf   = float(box.conf[0])
        if cls_id != 0 or conf < 0.3:
            continue
        x0, y0, x1, y1 = box.xyxy[0].cpu().numpy().astype(int)
        x0, y0 = max(0, x0), max(0, y0)
        x1, y1 = min(orig_W, x1), min(orig_H, y1)
        mask_person[y0:y1, x0:x1] = True

    mask_bg = ~(mask_text | mask_person)

    def resize_mask(m):
        return np.array(
            Image.fromarray(m.astype(np.uint8) * 255).resize((W, H), Image.NEAREST)
        ) > 127

    return {
        "text":       resize_mask(mask_text),
        "person":     resize_mask(mask_person),
        "background": resize_mask(mask_bg),
    }

def get_roi_masks_cached(image_path, H, W, cache):
    key = (image_path, H, W)
    if key not in cache:
        cache[key] = build_roi_masks(image_path, H, W)
    return cache[key]

def prebuild_roi_cache(sample_dfs, cache_file=ROI_CACHE_FILE):
    if os.path.exists(cache_file):
        print(f"Loading existing cache: {cache_file}")
        with open(cache_file, "rb") as f:
            return pickle.load(f)

    cache = {}
    all_paths = []
    for df in sample_dfs:
        all_paths.extend(df["resolved_path"].tolist())
    all_paths = list(dict.fromkeys(all_paths))

    print(f"Building ROI masks for {len(all_paths)} images...")
    for i, path in enumerate(all_paths):
        try:
            cache[(path, 224, 224)] = build_roi_masks(path, 224, 224)
            if (i + 1) % 10 == 0:
                print(f"  {i+1}/{len(all_paths)} done")
        except Exception as e:
            print(f"  WARNING {os.path.basename(path)}: {e}")

    with open(cache_file, "wb") as f:
        pickle.dump(cache, f)
    print(f"Cache saved: {cache_file}")
    return cache

# ── ROI Deletion / Insertion ──────────────────────────────────────────────────
def roi_deletion_insertion(image_path, roi_cache):
    img_orig = Image.open(image_path).convert("RGB")
    pv, rgb_np, H, W = get_pixel_values(img_orig)
    orig_probs = get_prob(pv)
    pred_class = int(np.argmax(orig_probs))
    orig_prob  = orig_probs[pred_class]

    blurred  = np.array(img_orig.resize((W, H)).filter(
        ImageFilter.GaussianBlur(radius=11)
    )).astype(np.float32) / 255.0
    mean_val = rgb_np.mean(axis=(0, 1), keepdims=True)

    roi_masks = get_roi_masks_cached(image_path, H, W, roi_cache)

    if not roi_masks["text"].any() and not roi_masks["person"].any():
        roi_masks["background"] = np.ones((H, W), dtype=bool)

    results = {}
    for roi_name, mask in roi_masks.items():
        m = mask[..., None].astype(np.float32)
        del_img = rgb_np * (1 - m) + mean_val * m
        del_pv, _, _, _ = get_pixel_values(Image.fromarray((del_img * 255).astype(np.uint8)))
        del_prob = float(get_prob(del_pv)[pred_class])

        ins_img = rgb_np * m + blurred * (1 - m)
        ins_pv, _, _, _ = get_pixel_values(Image.fromarray((ins_img * 255).astype(np.uint8)))
        ins_prob = float(get_prob(ins_pv)[pred_class])

        results[roi_name] = {"del": del_prob, "ins": ins_prob}

    return results, pred_class, float(orig_prob)

def batch_roi_analysis(sample_df, label, roi_cache, n_samples=50, save_path="roi_analysis.png"):
    sample_df = sample_df.head(n_samples).reset_index(drop=True)
    roi_del, roi_ins, orig_list, roi_pixel_ratios = {}, {}, {}, {}

    for _, row in sample_df.iterrows():
        try:
            img_orig = Image.open(row["resolved_path"]).convert("RGB")
            pv, rgb_np, H, W = get_pixel_values(img_orig)
            roi_masks = get_roi_masks_cached(row["resolved_path"], H, W, roi_cache)

            res, _, orig_p = roi_deletion_insertion(row["resolved_path"], roi_cache)
            orig_list.setdefault("vals", []).append(orig_p)

            for rname, vals in res.items():
                roi_del.setdefault(rname, []).append(vals["del"])
                roi_ins.setdefault(rname, []).append(vals["ins"])
                pixel_ratio = float(roi_masks[rname].mean()) if rname in roi_masks else 0.0
                roi_pixel_ratios.setdefault(rname, []).append(pixel_ratio)

        except Exception as e:
            print(f"  WARNING: skipping — {e}")

    if not roi_del:
        print("No valid samples"); return

    roi_names   = list(roi_del.keys())
    mean_orig   = np.mean(orig_list["vals"])
    mean_dels   = [np.mean(roi_del[r])          for r in roi_names]
    mean_ins_v  = [np.mean(roi_ins[r])          for r in roi_names]
    mean_ratios = [np.mean(roi_pixel_ratios[r]) for r in roi_names]

    eps      = 1e-6
    norm_del = [(mean_orig - mean_dels[i]) / (mean_ratios[i] + eps) for i in range(len(roi_names))]
    norm_ins = [mean_ins_v[i]              / (mean_ratios[i] + eps) for i in range(len(roi_names))]

    x      = np.arange(len(roi_names))
    colors = ["#e74c3c", "#3498db", "#2ecc71"]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))  # 높이 늘림

    for ax, norm_vals, raw_vals, direction, ylabel in [
        (axes[0], norm_del, mean_dels,
         f"Deletion Importance (prob drop / ROI area)\n[{label}]",   # title에 label 통합
         "Normalized importance [(orig - del_prob) / ROI_ratio]"),
        (axes[1], norm_ins, mean_ins_v,
         f"Insertion Importance (prob when only ROI visible / ROI area)\n[{label}]",
         "Normalized importance [ins_prob / ROI_ratio]"),
    ]:
        bars = ax.bar(x, norm_vals, 0.4, color=colors[:len(roi_names)], alpha=0.85)
        ax.set_title(direction, fontsize=10, pad=12)  # pad 여백 추가
        ax.set_xticks(x)
        ax.set_xticklabels(
            [f"{r}\n(area {mean_ratios[i]*100:.1f}%)" for i, r in enumerate(roi_names)],
            fontsize=10
        )
        ax.set_ylabel(ylabel, fontsize=9)
        ax.grid(axis="y", alpha=0.3)

        # y축 상단 여백 확보 — 숫자 레이블이 title과 안 겹치도록
        max_val = max(norm_vals) if norm_vals else 1
        ax.set_ylim(0, max_val * 1.25)

        for bar, nv, rv in zip(bars, norm_vals, raw_vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max_val * 0.02,  # 고정 offset 대신 비례 offset
                f"{nv:.2f}\n(raw:{rv:.3f})",
                ha="center", va="bottom", fontsize=9, fontweight="bold"
            )

    plt.suptitle(
        f"ROI Normalized Deletion / Insertion  [{label}]  (n={len(orig_list['vals'])} samples)\n"
        f"Mean baseline prob: {mean_orig:.3f}  |  Normalized = prob_change / ROI_area_ratio  |  "
        f"ROI detection: EasyOCR (text) + YOLOv8 (person)",
        fontsize=11,
        y=1.02  # suptitle을 subplot 위로 올림
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")
    plt.close(fig)

    print(f"\n  [{label}] Summary -- baseline mean: {mean_orig:.3f}")
    print(f"  {'ROI':<12}  area     Deletion  Insertion  Del-norm  Ins-norm")
    for i, rname in enumerate(roi_names):
        print(f"  {rname:<12}  {mean_ratios[i]*100:5.1f}%   {mean_dels[i]:.3f}     {mean_ins_v[i]:.3f}      {norm_del[i]:6.2f}     {norm_ins[i]:6.2f}")
        
# ── Batch Del/Ins curves ──────────────────────────────────────────────────────
def batch_deletion_insertion(sample_df, label, n_samples=50, n_steps=10, save_path="del_ins.png"):
    sample_df = sample_df.head(n_samples).reset_index(drop=True)
    all_del, all_ins, thrs = [], [], None

    for _, row in sample_df.iterrows():
        try:
            img_orig = Image.open(row["resolved_path"]).convert("RGB")
            pv, rgb_np, H, W = get_pixel_values(img_orig)
            orig_probs = get_prob(pv)
            pred_class = int(np.argmax(orig_probs))
            grayscale_cam = get_cam(pv, pred_class)
            thrs = np.linspace(0, 1, n_steps + 1)[1:]
            mean_val = rgb_np.mean(axis=(0, 1), keepdims=True)
            blurred  = np.array(img_orig.resize((W, H)).filter(
                ImageFilter.GaussianBlur(radius=11)
            )).astype(np.float32) / 255.0
            dels, ins = [], []
            for thr in thrs:
                mask = (grayscale_cam >= np.quantile(grayscale_cam, 1 - thr)).astype(np.float32)
                del_img = rgb_np * (1 - mask[..., None]) + mean_val * mask[..., None]
                del_pv, _, _, _ = get_pixel_values(Image.fromarray((del_img * 255).astype(np.uint8)))
                dels.append(float(get_prob(del_pv)[pred_class]))
                ins_img = rgb_np * mask[..., None] + blurred * (1 - mask[..., None])
                ins_pv, _, _, _ = get_pixel_values(Image.fromarray((ins_img * 255).astype(np.uint8)))
                ins.append(float(get_prob(ins_pv)[pred_class]))
            all_del.append(dels); all_ins.append(ins)
        except Exception as e:
            print(f"  WARNING: skipping — {e}")

    if not all_del:
        print("No valid samples"); return

    all_del = np.array(all_del); all_ins = np.array(all_ins)
    mean_del = all_del.mean(0); mean_ins = all_ins.mean(0)
    auc_del = float(np.trapezoid(mean_del, thrs))
    auc_ins = float(np.trapezoid(mean_ins, thrs))
    pct_labels = [f"{int(t*100)}%" for t in thrs]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, curves, mean_c, color1, color2, title, xlabel in [
        (axes[0], all_del, mean_del, "salmon", "red",
         f"Deletion Curve -- {label}\n(lower AUC = better)", "Fraction of image deleted"),
        (axes[1], all_ins, mean_ins, "skyblue", "blue",
         f"Insertion Curve -- {label}\n(higher AUC = better)", "Fraction of image revealed"),
    ]:
        for curve in curves:
            ax.plot(thrs, curve, color=color1, alpha=0.3, linewidth=0.8)
        auc = float(np.trapezoid(mean_c, thrs))
        ax.plot(thrs, mean_c, color=color2, linewidth=2.5, label=f"Mean (AUC={auc:.3f})")
        ax.set_title(title, fontsize=11)
        ax.set_xlabel(xlabel); ax.set_ylabel("Prediction probability")
        ax.set_xticks(thrs); ax.set_xticklabels(pct_labels, rotation=45)
        ax.legend(); ax.grid(alpha=0.3)

    plt.suptitle(
        f"Deletion / Insertion  [{label}]  n={len(all_del)}\n"
        f"Del AUC={auc_del:.3f}  |  Ins AUC={auc_ins:.3f}", fontsize=12
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}  |  Del AUC={auc_del:.4f}  Ins AUC={auc_ins:.4f}")
    plt.close(fig)

# ── Masking grid ──────────────────────────────────────────────────────────────
def masking_visualization_grid(sample_df, roi_cache, thresholds=(0.3, 0.5, 0.7), save_path="masking_grid.png"):
    """
    Columns: Original (+ROI overlay) | Heatmap | thr0.3 | thr0.5 | thr0.7
    ROI boundary colors: red=text, blue=person, green=background
    """
    sample_df = sample_df.reset_index(drop=True)
    n     = len(sample_df)
    ncols = 2 + len(thresholds)

    fig, axes = plt.subplots(n, ncols, figsize=(4 * ncols, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    ROI_COLORS = {"text":       [255, 80,  80],
                  "person":     [80,  120, 255],
                  "background": [80,  200, 80]}

    for i, (_, row) in enumerate(sample_df.iterrows()):
        img_orig = Image.open(row["resolved_path"]).convert("RGB")
        pv, rgb_np, H, W = get_pixel_values(img_orig)
        orig_probs    = get_prob(pv)
        pred_class    = int(np.argmax(orig_probs))
        grayscale_cam = get_cam(pv, pred_class)
        roi_masks     = get_roi_masks_cached(row["resolved_path"], H, W, roi_cache)

        overlay = (rgb_np * 255).astype(np.uint8).copy()
        for rname, rmask in roi_masks.items():
            color = np.array(ROI_COLORS[rname], dtype=np.uint8)
            overlay[rmask] = (overlay[rmask] * 0.6 + color * 0.4).astype(np.uint8)

        ch_name   = row.get("channel_name", "")
        vid_title = str(row.get("title", ""))[:28]
        axes[i, 0].imshow(overlay)
        axes[i, 0].set_title(
            f"pred={CLASS_NAMES[pred_class]} "
            f"({orig_probs[0]*100:.0f}%~34/{orig_probs[1]*100:.0f}%65~)\n"
            f"{ch_name} | {vid_title}\n"
            f"[red=text] [blue=person] [green=bg]", fontsize=6.5
        )
        axes[i, 0].axis("off")

        axes[i, 1].imshow(grayscale_cam, cmap="jet", vmin=0, vmax=1)
        axes[i, 1].set_title("Heatmap", fontsize=8)
        axes[i, 1].axis("off")

        for j, thr in enumerate(thresholds):
            mask   = grayscale_cam >= thr
            masked = rgb_np.copy()
            masked[~mask] = 0.0
            axes[i, 2 + j].imshow((masked * 255).astype(np.uint8))
            axes[i, 2 + j].set_title(f"thr>={thr:.1f} | {mask.mean()*100:.0f}%", fontsize=8)
            axes[i, 2 + j].axis("off")

    plt.suptitle(f"Masking Visualization ({n} images)  |  [red=text] [blue=person] [green=bg]", fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")
    plt.close(fig)

# ── Run all ───────────────────────────────────────────────────────────────────
ANALYSIS_N = 50

print("=" * 65)
print("[0] Building ROI mask cache (EasyOCR + YOLOv8)")
print("=" * 65)
roi_cache = prebuild_roi_cache(
    [correct_samples.head(ANALYSIS_N), correct_34_samples.head(ANALYSIS_N)]
)

print("\n" + "=" * 65)
print("[1] Deletion / Insertion curves")
print("=" * 65)
batch_deletion_insertion(correct_samples,    label="65~ correct",  n_samples=ANALYSIS_N, save_path="del_ins_65.png")
batch_deletion_insertion(correct_34_samples, label="~34 correct",  n_samples=ANALYSIS_N, save_path="del_ins_34.png")

print("\n" + "=" * 65)
print("[2] ROI-wise analysis (text / person / background)")
print("=" * 65)
batch_roi_analysis(correct_samples,    label="65~ correct",  roi_cache=roi_cache, n_samples=ANALYSIS_N, save_path="roi_analysis_65.png")
batch_roi_analysis(correct_34_samples, label="~34 correct",  roi_cache=roi_cache, n_samples=ANALYSIS_N, save_path="roi_analysis_34.png")

print("\n" + "=" * 65)
print("[3] Masking visualization grids (25 images x 2 files)")
print("=" * 65)
masking_visualization_grid(correct_samples.iloc[:25],    roi_cache, save_path="masking_65_grid_1.png")
masking_visualization_grid(correct_samples.iloc[25:50],  roi_cache, save_path="masking_65_grid_2.png")
masking_visualization_grid(correct_34_samples.iloc[:25], roi_cache, save_path="masking_34_grid_1.png")
masking_visualization_grid(correct_34_samples.iloc[25:], roi_cache, save_path="masking_34_grid_2.png")

print("\nAll analysis complete.")
print("  del_ins_65/34.png          -- Deletion/Insertion curves")
print("  roi_analysis_65/34.png     -- ROI bar charts")
print("  masking_65/34_grid_1/2.png -- Masking grids (with ROI overlay)")


Initializing EasyOCR (Korean + English)...
Initializing YOLOv8...
Models loaded
[0] Building ROI mask cache (EasyOCR + YOLOv8)
Building ROI masks for 100 images...
  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done
Cache saved: roi_masks_cache.pkl

[1] Deletion / Insertion curves
Saved: del_ins_65.png  |  Del AUC=0.7261  Ins AUC=0.7760
Saved: del_ins_34.png  |  Del AUC=0.7243  Ins AUC=0.8616

[2] ROI-wise analysis (text / person / background)
Saved: roi_analysis_65.png

  [65~ correct] Summary -- baseline mean: 0.997
  ROI           area     Deletion  Insertion  Del-norm  Ins-norm
  text           17.7%   0.940     0.686        0.32       3.89
  person         19.3%   0.805     0.785        0.99       4.06
  background     65.1%   0.698     0.809        0.46       1.24
Saved: roi_analysis_34.png

  [~34 correct] Summary -- baseline mean: 1.000
  ROI           area     Deletion  Insertion  Del-norm

/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 128149 (\N{TWO HEARTS}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/1138280325.py:378: UserWarning: Glyph 128149 (\N{TWO HEARTS}) missing from font(s) NanumGothic.
  plt.savefig(save_path, dpi=150, bbox_inches="tight")


Saved: masking_65_grid_2.png


/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 127807 (\N{HERB}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 129386 (\N{SANDWICH}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 129365 (\N{CARROT}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 129395 (\N{FACE WITH PARTY HORN AND PARTY HAT}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 127808 (\N{FOUR LEAF CLOVER}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 129325 (\N{SMILING FACE WITH SMILING EYES AND HAND COVERING MOUTH}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 128522 (\N{SMILING FACE WITH SMILING EYES}) 

Saved: masking_34_grid_1.png


/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 129293 (\N{WHITE HEART}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 128020 (\N{CHICKEN}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 129367 (\N{GREEN SALAD}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 129529 (\N{BROOM}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 128587 (\N{HAPPY PERSON RAISING ONE HAND}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 8205 (\N{ZERO WIDTH JOINER}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/1138280325.py:377: UserWarning: Glyph 65039 (\N{VARIATION SELECTOR-16}) missing from font(s) NanumGothic.
  plt.tight_layout()

Saved: masking_34_grid_2.png

All analysis complete.
  del_ins_65/34.png          -- Deletion/Insertion curves
  roi_analysis_65/34.png     -- ROI bar charts
  masking_65/34_grid_1/2.png -- Masking grids (with ROI overlay)


In [12]:
# =============================================================================
# ROI Detection Check -- EasyOCR (text) + YOLOv8 (person) visualization
# Columns: Original | Text mask | Person mask | Background mask | Composite
# =============================================================================
import matplotlib.patches as mpatches

def show_roi_detection(sample_df, roi_cache, n_show=10, save_path="roi_detection_check.png"):
    sample_df = sample_df.head(n_show).reset_index(drop=True)
    ncols = 5
    fig, axes = plt.subplots(n_show, ncols, figsize=(ncols * 4, n_show * 4))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    COL_TITLES = ["Original", "Text (red)", "Person (blue)", "Background (green)", "ROI composite"]
    for j, title in enumerate(COL_TITLES):
        axes[0, j].set_title(title, fontsize=10, fontweight="bold", pad=8)

    for i, (_, row) in enumerate(sample_df.iterrows()):
        img_orig = Image.open(row["resolved_path"]).convert("RGB")
        pv, rgb_np, H, W = get_pixel_values(img_orig)
        roi_masks = get_roi_masks_cached(row["resolved_path"], H, W, roi_cache)

        text_mask   = roi_masks["text"]
        person_mask = roi_masks["person"]
        bg_mask     = roi_masks["background"]

        text_pct   = text_mask.mean()   * 100
        person_pct = person_mask.mean() * 100
        bg_pct     = bg_mask.mean()     * 100

        ch_name   = row.get("channel_name", "")
        vid_title = str(row.get("title", ""))[:25]

        # Original
        axes[i, 0].imshow((rgb_np * 255).astype(np.uint8))
        axes[i, 0].set_title(f"{ch_name}\n{vid_title}", fontsize=6.5)
        axes[i, 0].axis("off")

        # Text mask
        text_vis = (rgb_np * 255).astype(np.uint8).copy()
        text_vis[text_mask]  = (text_vis[text_mask]  * 0.4 + np.array([255, 80, 80])  * 0.6).astype(np.uint8)
        text_vis[~text_mask] = (text_vis[~text_mask] * 0.3).astype(np.uint8)
        axes[i, 1].imshow(text_vis)
        axes[i, 1].set_title(f"detected: {text_pct:.1f}%", fontsize=8)
        axes[i, 1].axis("off")

        # Person mask
        person_vis = (rgb_np * 255).astype(np.uint8).copy()
        person_vis[person_mask]  = (person_vis[person_mask]  * 0.4 + np.array([80, 120, 255]) * 0.6).astype(np.uint8)
        person_vis[~person_mask] = (person_vis[~person_mask] * 0.3).astype(np.uint8)
        axes[i, 2].imshow(person_vis)
        detected = "found" if person_mask.any() else "none"
        axes[i, 2].set_title(f"{detected} ({person_pct:.1f}%)", fontsize=8)
        axes[i, 2].axis("off")

        # Background mask
        bg_vis = (rgb_np * 255).astype(np.uint8).copy()
        bg_vis[bg_mask]  = (bg_vis[bg_mask]  * 0.4 + np.array([80, 200, 80]) * 0.6).astype(np.uint8)
        bg_vis[~bg_mask] = (bg_vis[~bg_mask] * 0.3).astype(np.uint8)
        axes[i, 3].imshow(bg_vis)
        axes[i, 3].set_title(f"bg: {bg_pct:.1f}%", fontsize=8)
        axes[i, 3].axis("off")

        # Composite
        composite = (rgb_np * 255).astype(np.uint8).copy()
        composite[text_mask]   = (composite[text_mask]   * 0.45 + np.array([255, 80,  80])  * 0.55).astype(np.uint8)
        composite[person_mask] = (composite[person_mask] * 0.45 + np.array([80,  120, 255]) * 0.55).astype(np.uint8)
        composite[bg_mask]     = (composite[bg_mask] * 0.75).astype(np.uint8)
        axes[i, 4].imshow(composite)
        axes[i, 4].set_title(
            f"text {text_pct:.0f}% | person {person_pct:.0f}% | bg {bg_pct:.0f}%",
            fontsize=7.5
        )
        axes[i, 4].axis("off")

    legend_patches = [
        mpatches.Patch(color=(1.0, 0.31, 0.31), label="text"),
        mpatches.Patch(color=(0.31, 0.47, 1.0),  label="person"),
        mpatches.Patch(color=(0.31, 0.78, 0.31), label="background"),
    ]
    fig.legend(handles=legend_patches, loc="lower center", ncol=3,
               fontsize=10, frameon=True, bbox_to_anchor=(0.5, 0.0))

    plt.suptitle(
        f"ROI Detection Check  (EasyOCR + YOLOv8n)  --  {n_show} samples",
        fontsize=13, y=1.01
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")
    plt.close(fig)


# Run: 10 samples each for 65~ and ~34
print("[65~ correct samples] ROI detection check (10 images)")
show_roi_detection(correct_samples,    roi_cache, n_show=10, save_path="roi_check_65.png")

print("\n[~34 correct samples] ROI detection check (10 images)")
show_roi_detection(correct_34_samples, roi_cache, n_show=10, save_path="roi_check_34.png")

print("\nDone.")
print("  roi_check_65.png -- 65~ sample ROI detection")
print("  roi_check_34.png -- ~34 sample ROI detection")
print("  If detection looks wrong, delete roi_masks_cache.pkl and re-run.")


[65~ correct samples] ROI detection check (10 images)
Saved: roi_check_65.png

[~34 correct samples] ROI detection check (10 images)


/tmp/ipykernel_317481/769378247.py:88: UserWarning: Glyph 127807 (\N{HERB}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/769378247.py:88: UserWarning: Glyph 129386 (\N{SANDWICH}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/769378247.py:88: UserWarning: Glyph 129365 (\N{CARROT}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/769378247.py:88: UserWarning: Glyph 129395 (\N{FACE WITH PARTY HORN AND PARTY HAT}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/769378247.py:88: UserWarning: Glyph 127808 (\N{FOUR LEAF CLOVER}) missing from font(s) NanumGothic.
  plt.tight_layout()
/tmp/ipykernel_317481/769378247.py:89: UserWarning: Glyph 127807 (\N{HERB}) missing from font(s) NanumGothic.
  plt.savefig(save_path, dpi=130, bbox_inches="tight")
/tmp/ipykernel_317481/769378247.py:89: UserWarning: Glyph 129386 (\N{SANDWICH}) missing from font(s) NanumGothic.
  plt.savefig(save

Saved: roi_check_34.png

Done.
  roi_check_65.png -- 65~ sample ROI detection
  roi_check_34.png -- ~34 sample ROI detection
  If detection looks wrong, delete roi_masks_cache.pkl and re-run.
